In [69]:
import pandas as pd
from sqlalchemy import create_engine


engine = create_engine('mysql+pymysql://root:@localhost/review_analysis?charset=utf8mb4')

# cleaned_products 불러오기
cp = pd.read_sql("SELECT * FROM cleaned_products", engine)

# cleaned_reviews 불러오기
cr = pd.read_sql("SELECT * FROM cleaned_reviews", engine)

# df_v1 불러오기
df = pd.read_sql("SELECT * FROM df_v1", engine)

print("cleaned_products:", cp.shape)
print("cleaned_reviews:", cr.shape)
print("df_v1:", df.shape)

cleaned_products: (2948, 17)
cleaned_reviews: (685292, 48)
df_v1: (685292, 61)


In [72]:
import re
import pandas as pd
import numpy as np

def profile_text_data(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return "EMPTY"
    
    # 공백 제거 후 분석
    text_sub = text.strip()
    
    # 1. 언어 판별
    has_korean = bool(re.search('[가-힣]', text_sub))
    has_english = bool(re.search('[a-zA-Z]', text_sub))
    has_chinese = bool(re.search('[\u4e00-\u9fff]', text_sub)) # 중국어 한자
    
    # 2. 노이즈 판별
    has_number_only = text_sub.isdigit()
    # 특수문자/이모지만 있는 경우 (한글, 영어, 숫자 제외한 나머지만 있을 때)
    only_special = not bool(re.search('[가-힣a-zA-Z0-9]', text_sub))
    
    # 3. 종합 유형 분류
    if has_korean:
        if len(text_sub) < 5: return "KOREAN_SHORT (under 5 chars)"
        return "KOREAN_NORMAL"
    elif has_chinese:
        return "CHINESE_ONLY"
    elif has_english:
        # 영어 중에서도 난타(Gibberish) 판별: 공백 없는 긴 단어 포함 시
        if any(len(word) > 15 for word in text_sub.split()):
            return "ENGLISH_GIBBERISH"
        return "ENGLISH_ONLY"
    elif has_number_only:
        return "NUMBERS_ONLY"
    elif only_special:
        return "SPECIAL_CHARS_ONLY"
    else:
        return "UNKNOWN_NOISE"

# 데이터 진단 실행
df['profile_type'] = df['리뷰내용'].apply(profile_text_data)

# 결과 리포트 출력
report = df['profile_type'].value_counts().to_frame('count')
report['percentage'] = (report['count'] / len(df) * 100).round(2)

print("--- [데이터 종합 진단 리포트] ---")
print(report)

print("\n--- [유형별 실제 샘플 확인] ---")
for p_type in df['profile_type'].unique():
    print(f"\n[{p_type}] 샘플:")
    sample = df[df['profile_type'] == p_type]['리뷰내용'].head(3).tolist()
    for s in sample:
        print(f"  - {s}")

--- [데이터 종합 진단 리포트] ---
                               count  percentage
profile_type                                    
KOREAN_NORMAL                 684801       99.93
EMPTY                            250        0.04
ENGLISH_ONLY                     121        0.02
KOREAN_SHORT (under 5 chars)      57        0.01
ENGLISH_GIBBERISH                 38        0.01
NUMBERS_ONLY                      11        0.00
CHINESE_ONLY                       8        0.00
SPECIAL_CHARS_ONLY                 5        0.00
UNKNOWN_NOISE                      1        0.00

--- [유형별 실제 샘플 확인] ---

[KOREAN_NORMAL] 샘플:
  - 핏도 좋고 다 좋은데 바지 끝에 박음질이 엉성한 곳이 있네요 다시 교환하는 비용이랑  수선하는 비용이 별  차이 없을 것 같아서 걍 입을렵니다... ㅡㅡ 그런 부분좀 신경쓰셨으면...
  - 오굳굳 생각보다색상이쁘고 진짜편하네요.  앞으로조이쁜옷많이부탁드려여!
  - 재질이 뭔가 약간 면느낌보단 스포츠웨어 느낌이 난다 그냥 싼맛에 입는다

[KOREAN_SHORT (under 5 chars)] 샘플:
  - 굳
  - 여름
  - 예뻐용

[ENGLISH_ONLY] 샘플:
  - oh holy shiiiit this best cargo bazi
  - very good very good gameverygood
  - I love this one. I want to buy another 

In [73]:
df.columns

Index(['goodsNo', '플랫폼', '카테고리', '브랜드', '상품명', '정가', '판매가', '할인율', '조회수',
       '누적판매수', '리뷰수', '리뷰점수', '판매상태', 'sales_count_suspect',
       'sales_count_clean', 'view_count_suspect', 'inactive_candidate', '리뷰번호',
       '리뷰타입', '작성자', '리뷰내용', '평점', '체험단', '구매옵션', '키', '몸무게', '성별', '작성일',
       '만족도', '사진유무', '도움돼요', '구매사이즈', '구매상세', '연도', '월', '일', '요일', '평점_raw',
       '만족도_응답여부', '사이즈', '화면대비색감', '퀄리티', '구김', '두께감', '신축성', '색감', '보온성',
       '퀄리티_점수', '보온성_점수', '신축성_점수', '두께감_점수', '구김_점수', '사이즈_편차', '사이즈_편차절대',
       '화면대비색감_편차', '화면대비색감_편차절대', '색감_편차', '색감_편차절대', '노출일수', '일평균_도움돼요지수',
       '도움여부', 'profile_type'],
      dtype='str')

In [74]:
# 1. SPECIAL_CHARS_ONLY 유형 중 '....'이 포함된 데이터만 필터링
# . 이 3개 이상 연속되는 경우를 찾는 정규식입니다.
dots_reviews = df[
    (df['profile_type'] == 'SPECIAL_CHARS_ONLY') & 
    (df['리뷰내용'].str.contains(r'\.{3,}', na=False))
]

# 2. 보고 싶어 하시는 상품번호(goodsNo), 작성일자, 리뷰내용만 선택해서 출력
# 컬럼명은 실제 데이터프레임 구조에 맞춰 '작성일' 등으로 수정해서 사용하세요.
view_cols = ['카테고리', '브랜드', '상품명','리뷰타입','goodsNo', '작성자','작성일', '리뷰내용'] # 또는 'regDate' 등 날짜 컬럼명 확인 필요

# 데이터가 너무 많을 수 있으니 상위 30개 정도 먼저 확인
print("### 점(....)으로만 구성된 리뷰 상세 내역 ###")
if not dots_reviews.empty:
    # 날짜순으로 정렬해서 보고 싶을 경우
    display_df = dots_reviews[view_cols].sort_values(by='작성일', ascending=False)
    print(display_df.head(30))
else:
    print("해당 조건의 리뷰가 없습니다.")

# 3. CSV로 따로 저장해서 엑셀로 자세히 보고 싶다면 아래 주석 해제
# display_df.to_csv('dots_review_check.csv', index=False, encoding='utf-8-sig')

### 점(....)으로만 구성된 리뷰 상세 내역 ###
      카테고리 브랜드                         상품명   리뷰타입  goodsNo   작성자  \
83773   상의  제멋  하운드 투스 체크 반팔셔츠 차콜 HSSS2207  style  1015135  밀레시안   

                       작성일                  리뷰내용  
83773  2020-05-19 21:21:12  ....................  


In [75]:
# 1. SPECIAL_CHARS_ONLY 유형 중 '....'이 포함된 데이터만 필터링
# . 이 3개 이상 연속되는 경우를 찾는 정규식입니다.
reviews_unkown = df[
    (df['profile_type'] == 'UNKNOWN_NOISE')
]

# 2. 보고 싶어 하시는 상품번호(goodsNo), 작성일자, 리뷰내용만 선택해서 출력
# 컬럼명은 실제 데이터프레임 구조에 맞춰 '작성일' 등으로 수정해서 사용하세요.
view_cols = ['카테고리', '브랜드', '상품명','리뷰타입','goodsNo', '평점','작성자','작성일', '리뷰내용',
             '구매사이즈','키', '몸무게', '성별'] # 또는 'regDate' 등 날짜 컬럼명 확인 필요

# 데이터가 너무 많을 수 있으니 상위 30개 정도 먼저 확인
print("### UNKNOWN_NOISE 리뷰 상세 내역 ###")
if not reviews_unkown.empty:
    # 날짜순으로 정렬해서 보고 싶을 경우
    display_df = reviews_unkown[view_cols].sort_values(by='작성일', ascending=False)
    print(display_df.head(30))
else:
    print("해당 조건의 리뷰가 없습니다.")

# 3. CSV로 따로 저장해서 엑셀로 자세히 보고 싶다면 아래 주석 해제
# display_df.to_csv('dots_review_check.csv', index=False, encoding='utf-8-sig')

### UNKNOWN_NOISE 리뷰 상세 내역 ###
      카테고리  브랜드                       상품명   리뷰타입  goodsNo   평점    작성자  \
54053   상의  트래블  캠프 바시티 피그먼트 스웨트셔츠 블루 섀도우  goods   897632  5.0  원태입니다   

                       작성일                              리뷰내용   구매사이즈      키  \
54053  2019-10-26 01:41:27  302 1159 6318 11302 1159 6318 11  MEDIUM  170.0   

        몸무게       성별  
54053  62.0  missing  


In [76]:
import re

def 진단_시스템_노이즈(리뷰):
    if not isinstance(리뷰, str): return "결측치(NaN)"
    
    결과 = []
    
    # 1. URL 포함 여부 (http, https, www 등)
    if re.search(r'http\S+|www\S+', 리뷰):
        결과.append("URL 포함")
        
    # 2. HTML 태그 포함 여부 ( <br>, <div> 등)
    if re.search(r'<.*?>', 리뷰):
        결과.append("HTML 태그")
        
    # 3. 엑셀/시스템 오류 문자열
    # #NAME?, #VALUE!, #REF!, #DIV/0!, #NULL! 등
    if re.search(r'#[A-Z0-9/]+[?!]', 리뷰):
        결과.append("엑셀 오류문구")
        
    # 4. 줄바꿈 및 탭이 너무 많은 경우 (3개 이상)
    if len(re.findall(r'[\n\r\t]', 리뷰)) >= 3:
        결과.append("과도한 줄바꿈")

    # 5. 특수 공백 (전각 공백 등 일반 스페이스가 아닌 것)
    if '\xa0' in 리뷰 or '\u200b' in 리뷰:
        결과.append("특수 공백(ZeroWidth)")

    if not 결과:
        return "깨끗함"
    return ", ".join(결과)

# 진단 실행
df['시스템노이즈'] = df['리뷰내용'].apply(진단_시스템_노이즈)

# 통계 확인
노이즈_리포트 = df[df['시스템노이즈'] != "깨끗함"]['시스템노이즈'].value_counts()

print("### [ 시스템 노이즈 진단 결과 ] ###")
print(노이즈_리포트)

# 실제 샘플 확인
print("\n### [ 노이즈별 실제 샘플 ] ###")
노이즈_유형들 = df[df['시스템노이즈'] != "깨끗함"]['시스템노이즈'].unique()
for 유형 in 노이즈_유형들[:5]: # 상위 5개 유형만
    샘플 = df[df['시스템노이즈'] == 유형][['goodsNo', '작성일', '리뷰내용']].head(2)
    print(f"\n👉 유형: {유형}")
    print(샘플)

### [ 시스템 노이즈 진단 결과 ] ###
시스템노이즈
결측치(NaN)            250
HTML 태그             234
특수 공백(ZeroWidth)     52
URL 포함                5
Name: count, dtype: int64

### [ 노이즈별 실제 샘플 ] ###

👉 유형: HTML 태그
     goodsNo                  작성일  \
149   234480  2015-11-28 00:07:11   
187   282220  2015-12-01 18:04:04   

                                                  리뷰내용  
149  딱맞으<br /> 바지도 편하고 딷듯하고 대만족임 엠사이즈도 생각보다크니까 ㅇ엠사이...  
187  불량배송받았네요<br /> 고객센터 전화하니 제멋번호없다고 게시물 남기라해서<br ...  

👉 유형: 특수 공백(ZeroWidth)
       goodsNo                  작성일  \
18145   876284  2018-11-22 13:10:25   
73694  1291017  2020-03-17 22:12:51   

                                                    리뷰내용  
18145               재질도 좋고 어디에든 입을 수 있어 좋은거 같아요   목이 따뜻!  
73694  배송 빠르고 어디에나 잘 어울려서 좋습니다. ​ 어디에나 잘 어울리고 특별히 모난 ...  

👉 유형: 결측치(NaN)
       goodsNo                  작성일 리뷰내용
38076   987149  2019-06-09 23:28:50  NaN
52891   850153  2019-10-18 18:13:24  NaN

👉 유형: URL 포함
       goodsNo                  작성일  \
63681   8976

In [77]:
import re
import pandas as pd

def final_step1_cleaner(text):
    # 1. 기초 결측치 및 타입 체크
    if pd.isna(text) or not isinstance(text, str):
        return None
    
    # 2. 시스템 노이즈 정제 (텍스트는 살림)
    # HTML 태그 제거 (단어 붙음 방지를 위해 공백 한 칸으로 치환)
    text = re.sub(r'<.*?>', ' ', text)
    # URL 제거
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # 특수 공백(ZeroWidth) 및 제로 너비 공백 제거
    text = re.sub(r'[\xa0\u200b\u200c\u200d\u200e\u200f]', ' ', text)
    # 줄바꿈, 탭 정리
    text = re.sub(r'[\n\r\t]', ' ', text)

    # 3. 텍스트 정규화
    # 한글, 영어, 숫자, 기본 문장부호만 남기고 제거
    # text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?~]', ' ', text)
    # 이모지 유지하는 수정 코드
    text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?~\U00010000-\U0010ffff\u2600-\u27BF]', ' ', text)
    # 반복 문자 축소 (3번 이상 -> 2번)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 4. 최종 필터링 (결정하신 기준 적용)
    # 기준 1: 한국어가 반드시 포함되어야 함
    if not re.search('[가-힣]', text):
        return None
    
    # 기준 2: 3글자 이상만 남김 (KOREAN_SHORT 중 1, 2글자 컷)
    if len(text) < 3:
        return None
        
    return text

# --- 적용 및 결과 확인 ---
# 기존 df에서 1단계 전처리 적용
df['cleaned_content'] = df['리뷰내용'].apply(final_step1_cleaner)

# 결과가 None이 아닌 것들만 모아서 df_v2 생성
df_v2 = df.dropna(subset=['cleaned_content']).copy()

print(f"--- 1단계 완료 보고 ---")
print(f"✅ 원래 데이터 개수: {len(df):,}개")
print(f"✅ 1단계 정제 후 개수: {len(df_v2):,}개")
print(f"✅ 삭제된 데이터(2자 이하 한국어 포함): {len(df) - len(df_v2):,}개")

--- 1단계 완료 보고 ---
✅ 원래 데이터 개수: 685,292개
✅ 1단계 정제 후 개수: 684,837개
✅ 삭제된 데이터(2자 이하 한국어 포함): 455개


In [78]:
import re
import pandas as pd

# ================================================================
# Step 1 전처리 진단 코드 (df_v2 기준 — 재전처리 없음)
# df_v2['리뷰내용']       : 원본 텍스트
# df_v2['cleaned_content']: 이미 처리된 텍스트
# ================================================================

def diagnose_step1_v2(df_v2, orig_col='리뷰내용', clean_col='cleaned_content'):

    print("=" * 65)
    print("Step 1 전처리 진단 리포트 (df_v2 기준)")
    print("=" * 65)

    total = len(df_v2)
    orig  = df_v2[orig_col].astype(str)
    clean = df_v2[clean_col].astype(str)

    # ── 0. 기초 통계 ───────────────────────────────────────────
    print(f"\n[0] 기초 통계")
    print(f"  df_v2 행 수 (필터 통과 기준) : {total:>10,}")

    orig_len  = orig.apply(len)
    clean_len = clean.apply(len)
    diff_len  = orig_len - clean_len

    print(f"\n  {'항목':<15}  {'원본':>10}  {'정제 후':>10}  {'감소량':>10}")
    print(f"  {'─'*47}")
    print(f"  {'최솟값':<15}  {orig_len.min():>10,}  {clean_len.min():>10,}  {diff_len.min():>+10,}")
    print(f"  {'중앙값':<15}  {orig_len.median():>10,.0f}  {clean_len.median():>10,.0f}  {diff_len.median():>+10,.0f}")
    print(f"  {'평균':<15}  {orig_len.mean():>10,.1f}  {clean_len.mean():>10,.1f}  {diff_len.mean():>+10,.1f}")
    print(f"  {'최댓값':<15}  {orig_len.max():>10,}  {clean_len.max():>10,}  {diff_len.max():>+10,}")

    changed   = (diff_len > 0).sum()
    unchanged = (diff_len == 0).sum()
    print(f"\n  텍스트 변경됨   : {changed:>10,}행  ({changed/total*100:.1f}%)")
    print(f"  텍스트 그대로   : {unchanged:>10,}행  ({unchanged/total*100:.1f}%)")

    # ── 1. 각 정제 항목이 실제로 얼마나 발동했는지 ────────────
    print(f"\n[1] 정제 항목별 발동 행 수 (원본 기준)")

    checks = {
        'HTML 태그 (<...>)'       : r'<.+?>',
        'URL (http/www)'          : r'http\S+|www\S+',
        '제로폭 공백'              : r'[\xa0\u200b\u200c\u200d\u200e\u200f]',
        '줄바꿈·탭 (\\n\\r\\t)'   : r'[\n\r\t]',
        '제거 대상 특수문자'       : r'[^가-힣a-zA-Z0-9\s.,!?~\n\r\t]',
        '반복 문자 (3회↑)'        : r'(.)\1{2,}',
    }
    for label, pattern in checks.items():
        cnt = orig.str.contains(pattern, regex=True, na=False).sum()
        print(f"  {label:<25}: {cnt:>10,}  ({cnt/total*100:.2f}%)")

    # ── 2. 길이 구간별 분포 변화 ──────────────────────────────
    print(f"\n[2] 텍스트 길이 구간별 분포 변화")

    bins   = [0, 10, 30, 50, 100, 200, 500, float('inf')]
    labels = ['~10', '11~30', '31~50', '51~100', '101~200', '201~500', '501~']

    orig_cut  = pd.cut(orig_len,  bins=bins, labels=labels, right=True)
    clean_cut = pd.cut(clean_len, bins=bins, labels=labels, right=True)

    dist = pd.DataFrame({
        '원본 행수'  : orig_cut.value_counts().sort_index(),
        '정제후 행수': clean_cut.value_counts().sort_index(),
    })
    dist['증감'] = dist['정제후 행수'] - dist['원본 행수']

    print(f"\n  {'길이 구간':<10}  {'원본':>10}  {'정제후':>10}  {'증감':>8}")
    print(f"  {'─'*42}")
    for idx, row in dist.iterrows():
        sign = '+' if row['증감'] >= 0 else ''
        print(f"  {str(idx):<10}  {int(row['원본 행수']):>10,}  {int(row['정제후 행수']):>10,}  {sign}{int(row['증감']):>7,}")

    # ── 3. 샘플: 가장 많이 바뀐 10건 ─────────────────────────
    print(f"\n[3] 정제 전후 변화 샘플 (감소량 상위 10건)")

    top_idx = diff_len.nlargest(10).index
    for idx in top_idx:
        o = str(df_v2.loc[idx, orig_col])[:120].replace('\n','↵').replace('\t','→')
        c = str(df_v2.loc[idx, clean_col])[:120]
        print(f"\n  [원본  {orig_len[idx]:>4}자] {o}")
        print(f"  [정제후 {clean_len[idx]:>4}자] {c}")
        print(f"  {'─'*60}")

    # ── 4. 샘플: 변화 없는 케이스 (정제 효과 없는 정상 텍스트) ──
    print(f"\n[4] 정제 효과 없는 정상 텍스트 샘플 (5건)")
    no_change_idx = diff_len[diff_len == 0].index[:5]
    for idx in no_change_idx:
        print(f"  {repr(str(df_v2.loc[idx, clean_col])[:80])}")

    # ── 최종 요약 ──────────────────────────────────────────────
    print(f"\n{'=' * 65}")
    print(f"최종 요약")
    print(f"{'=' * 65}")
    print(f"  df_v2 행 수            : {total:>10,}")
    print(f"  정제 과정에서 변경된 행 : {changed:>10,}  ({changed/total*100:.1f}%)")
    print(f"  평균 텍스트 감소량      : {diff_len.mean():>10.1f}자")
    print(f"  총 제거된 문자 수       : {diff_len.sum():>10,}자")
    print(f"{'=' * 65}")


# ── 실행 ──────────────────────────────────────────────────────
diagnose_step1_v2(df_v2)

Step 1 전처리 진단 리포트 (df_v2 기준)

[0] 기초 통계
  df_v2 행 수 (필터 통과 기준) :    684,837

  항목                       원본        정제 후         감소량
  ───────────────────────────────────────────────
  최솟값                       3           3          +0
  중앙값                      36          35          +0
  평균                     44.9        44.4        +0.5
  최댓값                   1,460       1,419        +233

  텍스트 변경됨   :    133,271행  (19.5%)
  텍스트 그대로   :    551,566행  (80.5%)

[1] 정제 항목별 발동 행 수 (원본 기준)
  HTML 태그 (<...>)          :        234  (0.03%)
  URL (http/www)           :          4  (0.00%)
  제로폭 공백                   :         97  (0.01%)
  줄바꿈·탭 (\n\r\t)           :          0  (0.00%)
  제거 대상 특수문자               :     99,774  (14.57%)


/var/folders/6y/qvnf_sw97x1_0r6dbbkwmpc80000gn/T/ipykernel_36305/2283466632.py:52: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cnt = orig.str.contains(pattern, regex=True, na=False).sum()


  반복 문자 (3회↑)              :     24,749  (3.61%)

[2] 텍스트 길이 구간별 분포 변화

  길이 구간               원본         정제후        증감
  ──────────────────────────────────────────
  ~10                157         203  +     46
  11~30          208,061     216,503  +  8,442
  31~50          316,380     311,776   -4,604
  51~100         132,797     129,977   -2,820
  101~200         23,854      22,946     -908
  201~500          3,356       3,212     -144
  501~               232         220      -12

[3] 정제 전후 변화 샘플 (감소량 상위 10건)

  [원본   609자] 171 / 63  상의 티 : [스마트스토어] 연남백화점 - 3900원              트리플에이 무지티 반팔티 기본 긴팔티셔츠              트리플에이: 1301 반팔 / 사이즈: M / 색상: 화
  [정제후  376자] 171 63 상의 티 스마트스토어 연남백화점 3900원 트리플에이 무지티 반팔티 기본 긴팔티셔츠 트리플에이 1301 반팔 사이즈 M 색상 화이트 상의 셔츠 제멋 300원 기획특가 하운드 투스 체크 반팔셔츠 카키 HS
  ────────────────────────────────────────────────────────────

  [원본   451자] 어울릴만한 사람 : 😌본인이 허리가 두껍다 사세요                              😌키가 크다 사세요 기장이 엄청 널널하고                         통이 두꺼워서 몸이 큰 사람한
  [정제후  236자

# 중복처리

In [79]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# df_v2 에서 준중복 후보 (24h 이내, 같은 사람+상품+옵션+평점) 추출
df_v2['작성일'] = pd.to_datetime(df_v2['작성일'])

quasi_key = ['goodsNo', '작성자', '구매옵션']
candidates = []

for _, grp in df_v2.groupby(quasi_key):
    if len(grp) < 2:
        continue
    grp = grp.sort_values('작성일')
    dates = grp['작성일'].tolist()
    # 24h 이내 페어만
    for i in range(len(grp)):
        for j in range(i+1, len(grp)):
            diff_h = abs((dates[j] - dates[i]).total_seconds()) / 3600
            if diff_h <= 24:
                candidates.append({
                    '작성자'  : grp.iloc[i]['작성자'],
                    'goodsNo' : grp.iloc[i]['goodsNo'],
                    '텍스트A' : grp.iloc[i]['cleaned_content'],
                    '텍스트B' : grp.iloc[j]['cleaned_content'],
                    '시간차(h)': round(diff_h, 2),
                })

cand_df = pd.DataFrame(candidates)
print(f"24h 이내 준중복 후보 페어 수: {len(cand_df)}")

# 코사인 유사도 계산
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3))
sims = []
for _, row in cand_df.iterrows():
    try:
        mat = vec.fit_transform([row['텍스트A'], row['텍스트B']])
        sims.append(round(float(cosine_similarity(mat[0], mat[1])[0][0]), 4))
    except:
        sims.append(0.0)

cand_df['유사도'] = sims

# 유사도 분포 확인
print("\n유사도 분포:")
print(cand_df['유사도'].describe())
print()
print(pd.cut(cand_df['유사도'],
             bins=[0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
             include_lowest=True).value_counts().sort_index())

# 구간별 샘플 직접 확인 (저유사도 구간 포함)
for low, high in [(0.0, 0.1), (0.1, 0.2), (0.2, 0.3), (0.3, 0.5),
                  (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]:
    sample = cand_df[(cand_df['유사도'] >= low) & (cand_df['유사도'] < high)].head(2)
    if len(sample) == 0:
        continue
    print(f"\n{'='*60}")
    print(f"유사도 {low}~{high} 샘플")
    print(f"{'='*60}")
    for _, row in sample.iterrows():
        print(f"  A: {row['텍스트A'][:80]}")
        print(f"  B: {row['텍스트B'][:80]}")
        print(f"  유사도: {row['유사도']} | 시간차: {row['시간차(h)']}h")
        print()

24h 이내 준중복 후보 페어 수: 158471

유사도 분포:
count    158471.000000
mean          0.568350
std           0.413316
min           0.000000
25%           0.143900
50%           0.519200
75%           1.000000
max           1.000000
Name: 유사도, dtype: float64

유사도
(-0.001, 0.3]    67324
(0.3, 0.5]       11343
(0.5, 0.6]        2446
(0.6, 0.7]        2007
(0.7, 0.8]        2093
(0.8, 0.9]        2714
(0.9, 1.0]       70544
Name: count, dtype: int64

유사도 0.0~0.1 샘플
  A: 178 72 이구요 편하게 입기 좋아요 . 날이 추워서 롤업하지는 않고있는데 롤업하면 더 멋지게 입을수있겠네요
  B: 엄청 편하고 입고있어욬 루즈핏이라서 후드와 함께 입고다니면 엄청 멋스럽네요 많이 파세요~
  유사도: 0.0803 | 시간차: 0.05h

  A: 휴일이 껴있음에도 빠른배송 감사합니다~ 옷 너무편하고 이뻐요 잘입을개요~!
  B: 배기 핏이 많이 심하진 않으나 배기 팬츠로 보는게 맞는거 같네요. 172 78 L사이즈 적당하구요!! 제멋 바지 두번째 구매 인데 일단 무엇보다
  유사도: 0.052 | 시간차: 2.23h


유사도 0.1~0.2 샘플
  A: 전에 입던바지라서 재주문했어요 좋아하는 바지라서 주문했는데 여전하네요
  B: 전에도 주문했었던 바지인데 몇년지나고 아직 있어서 주문해 봤어요
  유사도: 0.1986 | 시간차: 0.16h

  A: 겨울에 안에 내복하나 입고 입으면 완전 든든할거 같아요 핏도 너무 이쁘네용 잘입을게요!
  B: 우선 따뜻해서 너무좋구요 지금 입기 딱 좋은거같애요 핏도 너무 이뻐요 잘입을게용
  유

In [80]:
"""
2단계: 중복 처리
=============================================================
분류 우선순위 (높음 → 낮음):
  1. 완전중복    : 리뷰번호 포함 모든 컬럼 동일 (시스템 오류)
  2. 중복        : 리뷰번호 제외 주요 컬럼 동일 + 24h 이내
  3. 준중복      : 리뷰 내용만 다르고 나머지 동일 + 24h 이내 + cosine_sim >= 0.5
  4. 준중복_복수 : 동일 IDENTITY, 24h 이내, cosine_sim < 0.5 (모두 유효)
  5. 재구매      : 동일 작성자+상품, 24h 초과 or 구매옵션 다름 (모두 유효)
  6. 정상        : 위 어느 것에도 해당 안 되는 단독 리뷰

대표 선정 기준:
  - 중복    : cleaned_content 길이가 가장 긴 것 (대표=True, 나머지=False)
  - 준중복  : cleaned_content 길이가 가장 긴 것 (대표=True, 나머지=False)
  (준중복_복수·재구매·정상은 모두 유효 → 중복_대표여부 전부 True)

처리 그룹:
  - style     : 리뷰타입 == 'style'
  - non_style : 나머지 전체 (goods / photo / general / month / experience)

탈퇴자('-') 처리:
  - 작성자 == "'-" 인 행은 각 행을 별개의 사람으로 간주
  - 작성자_처리 컬럼에 고유 ID 부여 (탈퇴_000001 …)

추가 컬럼:
  - 작성자_처리   : 탈퇴자는 행 고유 ID, 일반 사용자는 작성자 원본 값
  - 리뷰그룹      : 'style' | 'non_style'
  - 중복_구분     : '완전중복' | '중복' | '준중복' | '준중복_복수' | '재구매' | '정상'
  - 중복_그룹ID   : 같은 중복 묶음끼리 동일 ID (정상은 NaN)
  - 중복_유사도   : cosine similarity 점수 (준중복에만 기록, 나머지 NaN)
  - 중복_대표여부 : 그룹 내 대표로 남길 리뷰 True / 제거 후보 False
                    (준중복_복수·재구매·정상은 모두 True)
=============================================================
"""

import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ───────────────────────────────────────────────────────────
# 파라미터
# ───────────────────────────────────────────────────────────
DASH_AUTHOR = "'-"                  # 탈퇴/닉변 식별값
WINDOW_HOURS = 24                   # 중복 판정 시간 창 (h)
QUASI_SIM_THRESHOLD = 0.5           # 준중복 판정 cosine similarity 하한
TFIDF_ANALYZER = "char_wb"          # 한국어에 char_wb (단어 내 char n-gram) 효과적
TFIDF_NGRAM = (2, 3)                # 2~3-gram

# 완전중복·중복·준중복 판정에 사용할 핵심 컬럼들
IDENTITY_COLS = ["goodsNo", "작성자_처리", "구매옵션"]
CONTENT_COL   = "cleaned_content"   # step1 에서 생성된 정제 텍스트
DATE_COL      = "작성일"


# ───────────────────────────────────────────────────────────
# 헬퍼 함수
# ───────────────────────────────────────────────────────────
def assign_dash_ids(df: pd.DataFrame) -> pd.DataFrame:
    """탈퇴자('-') 각 행을 고유 ID로 분리"""
    df = df.copy()
    df["작성자_처리"] = df["작성자"].astype(str)
    dash_idx = df.index[df["작성자"] == DASH_AUTHOR]
    for rank, idx in enumerate(dash_idx, start=1):
        df.at[idx, "작성자_처리"] = f"탈퇴_{rank:06d}"
    return df


def assign_review_group(df: pd.DataFrame) -> pd.DataFrame:
    """리뷰타입 → 그룹 분류"""
    df = df.copy()
    df["리뷰그룹"] = df["리뷰타입"].apply(
        lambda x: "style" if x == "style" else "non_style"
    )
    return df


def init_dup_cols(df: pd.DataFrame) -> pd.DataFrame:
    """중복 관련 컬럼 초기화"""
    df = df.copy()
    df["중복_구분"]     = "정상"
    df["중복_그룹ID"]   = pd.NA
    df["중복_유사도"]   = pd.NA
    df["중복_대표여부"] = True
    return df


def _group_id_prefix(group_name: str, kind: str, n: int) -> str:
    return f"{group_name[:3]}_{kind}_{n:06d}"


def _longest_rep(valid_window: list, df: pd.DataFrame) -> object:
    """cleaned_content 길이가 가장 긴 행을 대표로 선택"""
    return max(valid_window, key=lambda x: len(str(df.at[x, CONTENT_COL] or "")))


# ───────────────────────────────────────────────────────────
# 완전중복 처리
# ───────────────────────────────────────────────────────────
# 완전중복 = 리뷰번호 포함 모든 컬럼이 100% 동일 (시스템 오류)
# 리뷰번호는 시스템 고유 ID이므로 실제 발생 건수는 0건이 정상
EXACT_COLS = ["리뷰번호"] + IDENTITY_COLS + [DATE_COL, CONTENT_COL]

def mark_exact_duplicates(df: pd.DataFrame, group_name: str, counter_start: int = 0):
    """
    완전중복: 리뷰번호를 포함한 모든 컬럼이 동일한 경우만 해당 (시스템 오류)
    리뷰번호가 고유 ID이므로 정상적으로는 0건
    """
    use_cols = [c for c in EXACT_COLS if c in df.columns]
    dup_mask = df.duplicated(subset=use_cols, keep=False)
    if not dup_mask.any():
        return df, counter_start

    df = df.copy()
    groups = df[dup_mask].groupby(use_cols, dropna=False)
    counter = counter_start
    for _, grp in groups:
        gid = _group_id_prefix(group_name, "exact", counter)
        counter += 1
        idxs = grp.sort_values("리뷰번호").index
        df.loc[idxs, "중복_구분"]      = "완전중복"
        df.loc[idxs, "중복_그룹ID"]    = gid
        df.loc[idxs[0],  "중복_대표여부"] = True
        df.loc[idxs[1:], "중복_대표여부"] = False

    return df, counter


# ───────────────────────────────────────────────────────────
# 중복 처리 (리뷰 내용 동일, 24h 이내)
# ───────────────────────────────────────────────────────────
DUP_KEY_COLS = IDENTITY_COLS + [CONTENT_COL]

def mark_duplicates(df: pd.DataFrame, group_name: str, counter_start: int = 0):
    """
    중복: IDENTITY + 리뷰내용 동일, 24h 이내
    대표: cleaned_content 길이가 가장 긴 것
    이미 완전중복 처리된 행은 건너뜀
    """
    remain = df[df["중복_구분"] == "정상"].copy()
    use_cols = [c for c in DUP_KEY_COLS if c in remain.columns]

    remain["_dup_key"] = remain[use_cols].apply(
        lambda r: tuple(str(v) for v in r), axis=1
    )
    groups = remain.groupby("_dup_key", dropna=False)

    df = df.copy()
    counter = counter_start
    for _, grp in groups:
        if len(grp) < 2:
            continue
        grp = grp.sort_values(DATE_COL)

        processed = set()
        for i, idx_i in enumerate(grp.index):
            if idx_i in processed:
                continue
            window = [idx_i]
            t_i = grp.at[idx_i, DATE_COL]
            for idx_j in list(grp.index)[i + 1:]:
                if idx_j in processed:
                    continue
                t_j = grp.at[idx_j, DATE_COL]
                if abs((t_j - t_i).total_seconds()) / 3600 <= WINDOW_HOURS:
                    window.append(idx_j)

            if len(window) >= 2:
                valid_window = [x for x in window if df.at[x, "중복_구분"] == "정상"]
                if len(valid_window) >= 2:
                    gid = _group_id_prefix(group_name, "dup", counter)
                    counter += 1
                    rep = _longest_rep(valid_window, df)   # ← 가장 긴 텍스트 대표
                    for x in valid_window:
                        df.at[x, "중복_구분"]     = "중복"
                        df.at[x, "중복_그룹ID"]   = gid
                        df.at[x, "중복_대표여부"] = (x == rep)
                    processed.update(valid_window)

    return df, counter


# ───────────────────────────────────────────────────────────
# 준중복 + 복수리뷰(24h 이내 저유사도) 처리
# ───────────────────────────────────────────────────────────
QUASI_KEY_COLS = IDENTITY_COLS  # 리뷰 내용 제외

def mark_quasi_duplicates(df: pd.DataFrame, group_name: str, counter_start: int = 0):
    """
    같은 IDENTITY 컬럼, 24h 이내 묶음을 대상으로:
      - cosine_sim >= QUASI_SIM_THRESHOLD → 준중복 (대표 1개, 나머지 제거 후보)
      - cosine_sim <  QUASI_SIM_THRESHOLD → 복수리뷰 (모두 유효)
    이미 완전중복·중복 처리된 행은 건너뜀
    """
    remain = df[df["중복_구분"] == "정상"].copy()
    use_cols = [c for c in QUASI_KEY_COLS if c in remain.columns]

    remain["_quasi_key"] = remain[use_cols].apply(
        lambda r: tuple(str(v) for v in r), axis=1
    )
    groups = remain.groupby("_quasi_key", dropna=False)

    df = df.copy()
    counter = counter_start
    vec = TfidfVectorizer(analyzer=TFIDF_ANALYZER, ngram_range=TFIDF_NGRAM, min_df=1)

    for _, grp in groups:
        if len(grp) < 2:
            continue
        grp = grp.sort_values(DATE_COL)

        processed = set()
        idxs = list(grp.index)

        for i, idx_i in enumerate(idxs):
            if idx_i in processed:
                continue
            t_i = grp.at[idx_i, DATE_COL]
            window = [idx_i]

            for idx_j in idxs[i + 1:]:
                if idx_j in processed:
                    continue
                t_j = grp.at[idx_j, DATE_COL]
                if abs((t_j - t_i).total_seconds()) / 3600 <= WINDOW_HOURS:
                    window.append(idx_j)

            if len(window) < 2:
                continue

            valid_window = [x for x in window if df.at[x, "중복_구분"] == "정상"]
            if len(valid_window) < 2:
                continue

            texts = [str(df.at[x, CONTENT_COL] or "") for x in valid_window]

            try:
                mat = vec.fit_transform(texts)
                sim_matrix = cosine_similarity(mat)
            except Exception:
                sim_matrix = np.zeros((len(valid_window), len(valid_window)))

            base = 0
            high_sim_indices = [base]   # 유사도 높은 묶음 (준중복)
            low_sim_indices  = []       # 유사도 낮은 묶음 (복수리뷰)

            for k in range(1, len(valid_window)):
                sim = float(sim_matrix[base, k])
                if sim >= QUASI_SIM_THRESHOLD:
                    high_sim_indices.append(k)
                else:
                    low_sim_indices.append(k)

            # ── 준중복 처리 (유사도 높은 묶음) ──────────────
            if len(high_sim_indices) >= 2:
                gid = _group_id_prefix(group_name, "quasi", counter)
                counter += 1
                actual_idxs = [valid_window[k] for k in high_sim_indices]
                rep = _longest_rep(actual_idxs, df)   # ← 가장 긴 텍스트 대표
                for k, x in zip(high_sim_indices, actual_idxs):
                    df.at[x, "중복_구분"]     = "준중복"
                    df.at[x, "중복_그룹ID"]   = gid
                    df.at[x, "중복_유사도"]   = round(float(sim_matrix[base, k]), 4)
                    df.at[x, "중복_대표여부"] = (x == rep)
                processed.update(actual_idxs)

            # ── 준중복_복수 처리 (유사도 낮은 묶음, 24h 이내) ──
            # base도 준중복에 포함되지 않은 경우 함께 묶음
            if len(high_sim_indices) < 2:
                low_sim_indices = [base] + low_sim_indices  # base도 미처리 상태

            low_actual = [
                valid_window[k] for k in low_sim_indices
                if df.at[valid_window[k], "중복_구분"] == "정상"
            ]
            if len(low_actual) >= 2:
                gid = _group_id_prefix(group_name, "quasi_multi", counter)
                counter += 1
                for x in low_actual:
                    df.at[x, "중복_구분"]     = "준중복_복수"
                    df.at[x, "중복_그룹ID"]   = gid
                    df.at[x, "중복_대표여부"] = True   # 준중복_복수는 모두 유효
                processed.update(low_actual)

    return df, counter


# ───────────────────────────────────────────────────────────
# 재구매 처리 (24h 초과 or 구매옵션 다름)
# ───────────────────────────────────────────────────────────
REPURCHASE_KEY = ["goodsNo", "작성자_처리"]

def mark_repurchase(df: pd.DataFrame, group_name: str, counter_start: int = 0):
    """
    재구매:
      - 동일 작성자 + 동일 상품에서 아직 '정상'으로 남은 행이 2개 이상
      - 날짜 24h 초과 차이가 있거나 구매옵션이 다른 경우
      → 모두 '재구매'로 표시, 중복_대표여부는 모두 True (전부 유효)
    """
    remain = df[df["중복_구분"] == "정상"].copy()
    use_cols = [c for c in REPURCHASE_KEY if c in remain.columns]
    groups = remain.groupby(use_cols, dropna=False)

    df = df.copy()
    counter = counter_start
    for (goodsNo, writer), grp in groups:
        if len(grp) < 2:
            continue

        grp = grp.sort_values(DATE_COL)
        dates   = grp[DATE_COL].tolist()
        options = grp["구매옵션"].tolist()

        time_spread_h    = (dates[-1] - dates[0]).total_seconds() / 3600
        has_diff_option  = len(set(str(o) for o in options)) > 1

        if (time_spread_h > WINDOW_HOURS) or has_diff_option:
            gid = _group_id_prefix(group_name, "repurchase", counter)
            counter += 1
            for idx in grp.index:
                df.at[idx, "중복_구분"]     = "재구매"
                df.at[idx, "중복_그룹ID"]   = gid
                df.at[idx, "중복_대표여부"] = True

    return df, counter


# ───────────────────────────────────────────────────────────
# 그룹별 통합 실행
# ───────────────────────────────────────────────────────────
def process_group(df: pd.DataFrame, group_name: str) -> pd.DataFrame:
    mask = df["리뷰그룹"] == group_name
    sub  = df[mask].copy()

    print(f"\n  ▶ [{group_name}] 그룹 처리 시작 (행 수: {len(sub):,})")

    c = 0
    sub, c = mark_exact_duplicates(sub, group_name, c)
    print(f"    ✅ 완전중복: {(sub['중복_구분']=='완전중복').sum()}행")

    sub, c = mark_duplicates(sub, group_name, c)
    print(f"    ✅ 중복: {(sub['중복_구분']=='중복').sum()}행")

    sub, c = mark_quasi_duplicates(sub, group_name, c)
    print(f"    ✅ 준중복: {(sub['중복_구분']=='준중복').sum()}행")
    print(f"    ✅ 준중복_복수: {(sub['중복_구분']=='준중복_복수').sum()}행")

    sub, c = mark_repurchase(sub, group_name, c)
    print(f"    ✅ 재구매: {(sub['중복_구분']=='재구매').sum()}행")

    print(f"    ✅ 정상: {(sub['중복_구분']=='정상').sum()}행")

    df.update(sub[["중복_구분", "중복_그룹ID", "중복_유사도", "중복_대표여부"]])
    return df


# ───────────────────────────────────────────────────────────
# 메인 실행
# ───────────────────────────────────────────────────────────
def run_step2(df_v2: pd.DataFrame) -> pd.DataFrame:
    print("=" * 60)
    print("2단계: 중복 처리 시작")
    print("=" * 60)

    df_v2 = df_v2.copy()
    df_v2[DATE_COL] = pd.to_datetime(df_v2[DATE_COL])

    df_v2 = assign_dash_ids(df_v2)
    n_dash = df_v2["작성자_처리"].str.startswith("탈퇴_").sum()
    print(f"  ℹ️  탈퇴자('-') 행 {n_dash}개 → 각각 고유 ID 부여")

    df_v2 = assign_review_group(df_v2)
    print(f"  ℹ️  style: {(df_v2['리뷰그룹']=='style').sum():,}행 | "
          f"non_style: {(df_v2['리뷰그룹']=='non_style').sum():,}행")

    df_v2 = init_dup_cols(df_v2)

    for group_name in ["style", "non_style"]:
        df_v2 = process_group(df_v2, group_name)

    print("\n" + "=" * 60)
    print("2단계 완료 요약")
    print("=" * 60)
    summary = df_v2["중복_구분"].value_counts()
    for label, cnt in summary.items():
        pct = cnt / len(df_v2) * 100
        print(f"  {label:<10}: {cnt:>8,}행  ({pct:.2f}%)")
    print(f"\n  전체 행 수: {len(df_v2):,}")
    print(f"  중복_대표여부=True  (유효 리뷰)  : {df_v2['중복_대표여부'].sum():,}행")
    print(f"  중복_대표여부=False (제거 후보)  : {(~df_v2['중복_대표여부']).sum():,}행")

    return df_v2


# ───────────────────────────────────────────────────────────
# 직접 실행 시 테스트 (df_v1_data.csv → step1 → step2)
# ───────────────────────────────────────────────────────────
if __name__ == "__main__":
    import re as _re

    DATA_PATH  = "df_v1_data.csv"
    OUTPUT_PATH = "df_v2_dedup.csv"

    def final_step1_cleaner(text):
        if pd.isna(text) or not isinstance(text, str):
            return None
        text = _re.sub(r'<.*?>', ' ', text)
        text = _re.sub(r'http\S+|www\S+', ' ', text)
        text = _re.sub(r'[\xa0\u200b\u200c\u200d\u200e\u200f]', ' ', text)
        text = _re.sub(r'[\n\r\t]', ' ', text)
        text = _re.sub(r'[^가-힣a-zA-Z0-9\s.,!?~]', ' ', text)
        text = _re.sub(r'(.)\1{2,}', r'\1\1', text)
        text = _re.sub(r'\s+', ' ', text).strip()
        if not _re.search('[가-힣]', text):
            return None
        if len(text) < 3:
            return None
        return text

    print("데이터 로딩 중...")
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print(f"원본 데이터: {len(df):,}행")

    print("\nStep 1: 텍스트 정제 중...")
    df["cleaned_content"] = df["리뷰내용"].apply(final_step1_cleaner)
    df_v2 = df.dropna(subset=["cleaned_content"]).copy()
    print(f"Step 1 후: {len(df_v2):,}행 (삭제: {len(df)-len(df_v2):,}행)")

    print("\nStep 2: 중복 처리 중...")
    df_v3 = run_step2(df_v2)

    print(f"\n결과 저장 중: {OUTPUT_PATH}")
    df_v3.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print("저장 완료!")

    print("\n=== 준중복 샘플 ===")
    quasi_sample = df_v3[df_v3["중복_구분"] == "준중복"].head(6)
    if len(quasi_sample) > 0:
        print(quasi_sample[["리뷰번호", "작성자_처리", "goodsNo", "구매옵션",
                             "작성일", "리뷰그룹", "중복_그룹ID",
                             "중복_유사도", "중복_대표여부", "cleaned_content"]].to_string())
    else:
        print("  준중복 케이스 없음")

    print("\n=== 준중복_복수 샘플 ===")
    multi_sample = df_v3[df_v3["중복_구분"] == "준중복_복수"].head(6)
    if len(multi_sample) > 0:
        print(multi_sample[["리뷰번호", "작성자_처리", "goodsNo", "구매옵션",
                             "작성일", "리뷰그룹", "중복_그룹ID",
                             "중복_대표여부", "cleaned_content"]].to_string())

    print("\n=== 재구매 샘플 ===")
    repurchase_sample = df_v3[df_v3["중복_구분"] == "재구매"].head(6)
    if len(repurchase_sample) > 0:
        print(repurchase_sample[["리뷰번호", "작성자_처리", "goodsNo", "구매옵션",
                                  "작성일", "리뷰그룹", "중복_그룹ID",
                                  "중복_대표여부", "cleaned_content"]].to_string())


데이터 로딩 중...
원본 데이터: 685,292행

Step 1: 텍스트 정제 중...
Step 1 후: 684,837행 (삭제: 455행)

Step 2: 중복 처리 중...
2단계: 중복 처리 시작
  ℹ️  탈퇴자('-') 행 3974개 → 각각 고유 ID 부여
  ℹ️  style: 104,358행 | non_style: 580,479행

  ▶ [style] 그룹 처리 시작 (행 수: 104,358)
    ✅ 완전중복: 0행
    ✅ 중복: 115행
    ✅ 준중복: 26행
    ✅ 준중복_복수: 143행
    ✅ 재구매: 4812행
    ✅ 정상: 99262행

  ▶ [non_style] 그룹 처리 시작 (행 수: 580,479)
    ✅ 완전중복: 0행
    ✅ 중복: 59569행
    ✅ 준중복: 10032행
    ✅ 준중복_복수: 65855행
    ✅ 재구매: 92088행
    ✅ 정상: 352935행

2단계 완료 요약
  정상        :  452,197행  (66.03%)
  재구매       :   96,900행  (14.15%)
  준중복_복수    :   65,998행  (9.64%)
  중복        :   59,684행  (8.72%)
  준중복       :   10,058행  (1.47%)

  전체 행 수: 684,837
  중복_대표여부=True  (유효 리뷰)  : 649,920행
  중복_대표여부=False (제거 후보)  : 34,917행

결과 저장 중: df_v2_dedup.csv
저장 완료!

=== 준중복 샘플 ===
       리뷰번호    작성자_처리  goodsNo   구매옵션                 작성일       리뷰그룹           중복_그룹ID  중복_유사도  중복_대표여부                                           cleaned_content
409  414083  침산동TinyG   282220  백오트밀M 2015-